# Spark + Google Cloud Storage (PySpark on your Mac → read `gs://`)

This is the **personal GCP** path from [cloud.md](cloud.md): use **your** bucket instead of the course’s `dtc_data_lake_de-zoomcamp-nytaxi` bucket (you may not have access to that one).

**You will:** install the Hadoop GCS connector JAR, start a local Spark session that can talk to GCS, and read Parquet from `gs://your-bucket/...`.

## Prerequisites

1. **Google Cloud SDK:** `gcloud` and `gsutil` installed; logged in: `gcloud auth login`.
2. **Application Default Credentials** (for local dev): run once in a terminal:
   ```bash
   gcloud auth application-default login
   ```
3. **A bucket you own** (example below uses a placeholder — replace with yours).
4. **Java 17** and **PySpark** (same as Module 6 setup). If Spark fails with `JAVA_GATEWAY_EXITED`, set `JAVA_HOME` (e.g. Homebrew `openjdk@17`).

**Course video/docs:** [06-batch README](../README.md) and [cloud.md](cloud.md) (Dataproc, `spark-submit`, BigQuery come after this notebook).

## 1. Download the GCS connector JAR (once per machine)

From the repo folder `06-batch/code/`:

```bash
mkdir -p lib
gsutil cp gs://hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar lib/
```

Older courses used `gcs-connector-hadoop3-2.2.5.jar`; **`latest`** supports **Application Default Credentials** via `fs.gs.auth.type=APPLICATION_DEFAULT` (no service-account JSON file required for local learning).

## 2. Upload data (optional)

Example: copy Module 6 Yellow file into your bucket:

```bash
gsutil cp /path/to/yellow_tripdata_2025-11.parquet gs://YOUR_BUCKET/hw6/yellow_tripdata_2025-11.parquet
```

Or use any Parquet you already have in GCS.

In [ ]:
# --- Config: edit these ---
import os
from pathlib import Path

# Your GCP project id (gcloud config get-value project)
GCP_PROJECT = os.environ.get("GCP_PROJECT", "")

# Bucket name only (no gs://)
GCS_BUCKET = os.environ.get("GCS_BUCKET", "")

# Path to one Parquet file or a directory of Parquet parts inside the bucket
GCS_PARQUET = os.environ.get(
    "GCS_PARQUET",
    "yellow_tripdata_2024-01.parquet",  # change to hw6/... after you upload
)

GCS_URI = f"gs://{GCS_BUCKET}/{GCS_PARQUET}"

# This notebook lives in 06-batch/code/
CODE_DIR = Path.cwd()
if not (CODE_DIR / "lib").is_dir():
    CODE_DIR = Path.cwd().parent / "code" if (Path.cwd().parent / "code" / "lib").is_dir() else Path.cwd()

JAR = CODE_DIR / "lib" / "gcs-connector-hadoop3-latest.jar"
if not JAR.exists():
    alt = CODE_DIR / "lib" / "gcs-connector-hadoop3-2.2.5.jar"
    JAR = alt if alt.exists() else JAR

# macOS: Jupyter often misses JAVA_HOME
if not os.environ.get("JAVA_HOME"):
    for _home in (
        "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
        "/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
    ):
        if Path(_home, "bin", "java").exists():
            os.environ["JAVA_HOME"] = _home
            os.environ["PATH"] = f"{_home}/bin{os.pathsep}{os.environ.get('PATH', '')}"
            break

print("GCP_PROJECT:", GCP_PROJECT)
print("GCS_URI:", GCS_URI)
print("JAR exists:", JAR.exists(), JAR)

GCP_PROJECT: project-3a3cf597-6dcd-4db6-8c0
GCS_URI: gs://dezoomcamp-hw3-3a3cf597/yellow_tripdata_2024-01.parquet
JAR exists: True /Users/sahand/Desktop/DataTalks/DataEngineering2025/data-engineering-zoomcamp/06-batch/code/lib/gcs-connector-hadoop3-latest.jar


In [2]:
if not JAR.exists():
    raise FileNotFoundError(
        f"Missing GCS connector JAR. Run: mkdir -p lib && "
        f"gsutil cp gs://hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar {JAR.parent}/"
    )

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[*]")
    .appName("spark-gcs-local")
    .config("spark.jars", str(JAR.resolve()))
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config(
        "spark.hadoop.fs.AbstractFileSystem.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS",
    )
    .config("spark.hadoop.fs.gs.auth.type", "APPLICATION_DEFAULT")
    .config("spark.hadoop.fs.gs.project.id", GCP_PROJECT)
    .getOrCreate()
)

print("spark.version:", spark.version)

26/03/30 12:31:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 12:31:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


spark.version: 4.1.1


In [3]:
# Read Parquet from GCS (works like a local path, but gs://)
df = spark.read.parquet(GCS_URI)
print("rows:", df.count())
df.printSchema()
df.show(3)

26/03/30 12:33:49 WARN FileSystem: Failed to initialize filesystem gs://dezoomcamp-hw3-3a3cf597/yellow_tripdata_2024-01.parquet: java.io.IOException: Error getting access token from metadata server at: http://169.254.169.254/computeMetadata/v1/instance/service-accounts/default/token
26/03/30 12:33:49 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: gs://dezoomcamp-hw3-3a3cf597/yellow_tripdata_2024-01.parquet.
java.io.IOException: Error getting access token from metadata server at: http://169.254.169.254/computeMetadata/v1/instance/service-accounts/default/token
	at com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.util.CredentialFactory.getCredentialFromMetadataServiceAccount(CredentialFactory.java:254)
	at com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.util.CredentialFactory.getCredential(CredentialFactory.java:406)
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.getCredential(GoogleHa

Py4JJavaError: An error occurred while calling o40.parquet.
: java.io.IOException: Error getting access token from metadata server at: http://169.254.169.254/computeMetadata/v1/instance/service-accounts/default/token
	at com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.util.CredentialFactory.getCredentialFromMetadataServiceAccount(CredentialFactory.java:254)
	at com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.util.CredentialFactory.getCredential(CredentialFactory.java:406)
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.getCredential(GoogleHadoopFileSystemBase.java:1479)
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.createGcsFs(GoogleHadoopFileSystemBase.java:1638)
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.configure(GoogleHadoopFileSystemBase.java:1620)
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.initialize(GoogleHadoopFileSystemBase.java:507)
	at org.apache.hadoop.fs.FileSystem.createFileS

## 3. Write back to GCS (optional)

Example: repartition and write Parquet to a **new** prefix (you pay for storage).

```python
out = f"gs://{GCS_BUCKET}/spark-output/demo/"
df.repartition(2).write.mode("overwrite").parquet(out)
```

Uncomment and run in a new cell if you want to try it.

## 4. Next steps (real “cloud Spark”)

| Topic | Where |
|--------|--------|
| Upload folders, `gsutil` | [cloud.md](cloud.md) |
| Original course notebook (service-account JSON) | [09_spark_gcs.ipynb](09_spark_gcs.ipynb) |
| Dataproc / `gcloud dataproc jobs submit pyspark` | [cloud.md](cloud.md) |
| BigQuery + Spark | [cloud.md](cloud.md) + `06_spark_sql_big_query.py` |

Stop Spark when finished:

```python
spark.stop()
```